# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a structured guide to loading and exploring the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

In [ ]:
# Show the record sets in the dataset
print('Available Record Sets:')
for record_set in metadata.record_sets:
    print(f"- name: {record_set.name} | @id: {record_set.id}")
    print("  Fields/Columns:")
    for field in record_set.fields:
        print(f"    - {field.name} | @id: {field.id} | type: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

**Note:** All entities are referenced by their `@id`. Please check the list above for IDs. The main record set in this dataset is `cr:KilifiSurvey`.

In [ ]:
# Collect the record set IDs available
record_sets = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for Record Set ID: {record_set_id}")

# For this dataset, we expect typically one main record set. Let's display its columns and show the first few rows.
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id is not None:
    print(f"Fields in the Record Set ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In this step, we will:
- Select a numeric field, e.g., GAD-7 Score, by its field `@id`.
- Filter out records according to a value threshold.
- Normalize the selected numeric field.
- Group by a categorical field such as gender (again using its field `@id`).

In [ ]:
# Identify the main record set and numeric fields from earlier listing
# Example field IDs (replace with the exact @id from section 2 if different):
record_set_id = main_record_set_id  # e.g., 'cr:KilifiSurvey'
numeric_field_id = None  # Will auto-detect GAD-7/PHQ-9/PSQ@id
categorical_field_id = None  # Will auto-detect e.g. 'gender' or 'cr:gender'

if record_set_id is not None:
    df = dataframes[record_set_id]
    # Try to auto-detect common mental health score field IDs
    candidates = [c for c in df.columns if any(x in c.lower() for x in ['gad', 'phq', 'psq'])]
    if candidates:
        numeric_field_id = candidates[0]  # select first likely field
        print(f"Using numeric field: {numeric_field_id}")
    else:
        print("Could not find common mental health score field column. Please set 'numeric_field_id'.")

    # Try to find a group/categorical field (like gender)
    for c in df.columns:
        if 'gender' in c.lower() or 'sex' in c.lower():
            categorical_field_id = c
            print(f"Using group field: {categorical_field_id}")
            break
    if numeric_field_id is not None and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        if categorical_field_id is not None:
            grouped_df = filtered_df.groupby(categorical_field_id)[numeric_field_id].mean().to_frame()
            print(f"Grouped mean {numeric_field_id} by {categorical_field_id}:")
            display(grouped_df.head())
    else:
        print(f"Selected field '{numeric_field_id}' is not numeric or was not found.")
else:
    print("No main record set loaded; cannot proceed with EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id is not None and numeric_field_id is not None:
    sns.set(style="whitegrid")
    # Distribution of the selected numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot by group field if available
    if categorical_field_id is not None:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=categorical_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {categorical_field_id}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated how to load and explore the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

- The dataset includes demographic and mental health indicators (such as GAD-7, PHQ-9, and PSQ scores) from Kilifi County, Kenya.
- We listed all available record sets and fields using their `@id`s for traceability and reproducibility.
- Data extraction and simple exploratory analysis were performed, including filtering and normalization.
- Basic visualizations provided insights into score distributions and demographic group differences.

Further analysis could involve building ML models or performing advanced statistics on this AI-ready, FAIR-compliant dataset.